# 17 Deep Structured Sequence Features

Purpose: convert raw CV outputs from notebook 16 into `features/structured_sequence_v1/{manifest,frames}.parquet` for trainable sequence models.

Required inputs:

- `clips/{FULL_RUN_ID}/clips_v1.parquet`
- raw CV artifacts from notebook 16. Empty artifacts are accepted for smoke, but real mechanics features require detections/pose/bat/homography rows.

Created outputs:

- `features/structured_sequence_v1/manifest.parquet`
- `features/structured_sequence_v1/frames.parquet`


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)
print('STRUCTURED_SEQUENCE_FEATURE_ID =', STRUCTURED_SEQUENCE_FEATURE_ID)


In [ ]:
from sport_pipeline.artifact_check import check_artifacts

required = [
    f'clips/{FULL_RUN_ID}/clips_v1.parquet',
    f'detections/{FULL_RUN_ID}/detections_v1.parquet',
    f'pose2d/{FULL_RUN_ID}/pose2d_v1.parquet',
    f'objects/{FULL_RUN_ID}/bat_line_v1.parquet',
    f'homography/{FULL_RUN_ID}/homography_v1.parquet',
]
print(check_artifacts(BASE_DIR, required))


In [ ]:
from sport_pipeline.features.deep_sequence_builder import build_deep_sequence_artifacts

DEEP_SEQUENCE_SETTINGS = stage_settings(RUN_PROFILE, 'deep_sequence_features')
FRAME_COUNT = int(DEEP_SEQUENCE_SETTINGS.get('frame_count', 32))
MAX_CLIPS = DEEP_SEQUENCE_SETTINGS.get('max_clips')
REQUIRE_NON_EMPTY_SEQUENCES = bool(DEEP_SEQUENCE_SETTINGS.get('require_non_empty', True))
FORCE_REBUILD_DEEP_SEQUENCE = os.environ.get('FORCE_REBUILD_DEEP_SEQUENCE', '0').lower() in {'1', 'true', 'yes'}
RESUME_DEEP_SEQUENCE = bool(DEEP_SEQUENCE_SETTINGS.get('resume', True)) and not FORCE_REBUILD_DEEP_SEQUENCE
CHECKPOINT_EVERY_CLIPS = int(DEEP_SEQUENCE_SETTINGS.get('checkpoint_every_clips', 25))
print('deep_sequence resume =', RESUME_DEEP_SEQUENCE, 'force_rebuild =', FORCE_REBUILD_DEEP_SEQUENCE)

outputs = build_deep_sequence_artifacts(
    BASE_DIR,
    FULL_RUN_ID,
    frame_count=FRAME_COUNT,
    max_clips=MAX_CLIPS,
    require_non_empty=REQUIRE_NON_EMPTY_SEQUENCES,
    resume=RESUME_DEEP_SEQUENCE,
    checkpoint_every_clips=CHECKPOINT_EVERY_CLIPS,
    structured_sequence_feature_id=STRUCTURED_SEQUENCE_FEATURE_ID,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.io import read_table

expected = [
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/manifest.parquet',
    f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/frames.parquet',
    f'reports/preflight/deep_sequence_features_{FULL_RUN_ID}_progress.json',
]
print(check_artifacts(BASE_DIR, expected))
manifest_path = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'manifest.parquet'
frames_path = BASE_DIR / 'features' / STRUCTURED_SEQUENCE_FEATURE_ID / 'frames.parquet'
manifest_rows = read_table(manifest_path) if manifest_path.exists() else []
frame_rows = read_table(frames_path) if frames_path.exists() else []
print('structured_sequence manifest rows =', len(manifest_rows))
print('structured_sequence frame rows =', len(frame_rows))
if not manifest_rows or not frame_rows:
    raise RuntimeError('17 produced empty sequence artifacts. Do not continue to 18. Check 12 clips_v1 rows and rerun 17 after clips exist.')
print('NEXT: notebooks/18_sequence_tcn_training.ipynb')
